# QTran 三数据集公平比较实验

本 Notebook 对 WM-811K、MixedWM38 和 Carinthia 分别训练五量子投影 QTran 及三个参数匹配基线。三个数据集互不混合、检查点互不共用；模型选择只使用验证集，测试集仅在最佳验证检查点冻结后评价。

## 0. 使用顺序

1. 先运行路径与原始数据检查。
2. 三个缓存单元分别运行一次。
3. 用 `DATASET_TO_RUN` 每次选择一个数据集。
4. 先执行审计和单种子试跑，确认后再运行五种子。
5. 三个数据集全部完成后再运行总表、统计和论文绘图单元。

In [34]:
from dataclasses import replace
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import display

PROJECT_DIR = Path.cwd().resolve()
if not (PROJECT_DIR / 'qcs_wm811k.py').exists() and (PROJECT_DIR / 'autodl' / 'qcs_wm811k.py').exists():
    PROJECT_DIR = PROJECT_DIR / 'autodl'
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from qcs_core import ExperimentConfig, environment_report, summarize_results
from qcs_datasets import (
    class_distribution, load_dataset_cache, make_dataset_split,
    prepare_carinthia_cache, prepare_mixedwm38_cache,
    prepare_wm811k_cache, split_distribution,
)
from qcs_multidataset import (
    dataset_audit, evaluate_noise_suite, paired_model_comparisons,
    plot_cross_dataset_results, run_comparison_suite,
)

print('PROJECT_DIR:', PROJECT_DIR)
print(environment_report())

PROJECT_DIR: /root/xxx/autodl
{'torch': '2.7.1+cu118', 'deepquantum': '4.5.0', 'cuda_available': True, 'cuda_device': 'NVIDIA GeForce RTX 4090'}


In [34]:
RAW_PATHS = {
    'wm811k': PROJECT_DIR / 'data' / 'LSWMD.pkl',
    'mixedwm38': PROJECT_DIR / 'data' / 'raw' / 'mixedwm38' / 'Wafer_Map_Datasets.npz',
    'carinthia': PROJECT_DIR / 'data' / 'raw' / 'carinthia' / 'data.zip',
}
CACHE_PATHS = {
    'wm811k': PROJECT_DIR / 'data_cache' / 'wm811k_labeled_32.npz',
    'mixedwm38': PROJECT_DIR / 'data_cache' / 'mixedwm38_32.npz',
    'carinthia': PROJECT_DIR / 'data_cache' / 'carinthia_32.npz',
}
ARTIFACT_ROOT = PROJECT_DIR / 'artifacts' / 'three_datasets_five_projection'

path_rows = []
for name, path in RAW_PATHS.items():
    path_rows.append({
        'dataset': name, 'path': str(path), 'exists': path.exists(),
        'size_gb': path.stat().st_size / 1024**3 if path.exists() else None,
    })
display(pd.DataFrame(path_rows).round(3))
assert all(path.exists() for path in RAW_PATHS.values()), '存在缺失的原始数据文件'

KeyboardInterrupt: 

## 1. 分别生成缓存

原始文件只读。已有缓存时会立即返回；只有明确需要重建时才将 `force` 改为 `True`。

In [ ]:
prepare_wm811k_cache(
    RAW_PATHS['wm811k'], CACHE_PATHS['wm811k'], image_size=32, force=False
)

PosixPath('/root/xxx/autodl/data_cache/wm811k_labeled_32.npz')

In [ ]:
prepare_mixedwm38_cache(
    RAW_PATHS['mixedwm38'], CACHE_PATHS['mixedwm38'], image_size=32, force=False
)

PosixPath('/root/xxx/autodl/data_cache/mixedwm38_32.npz')

In [ ]:
prepare_carinthia_cache(
    RAW_PATHS['carinthia'], CACHE_PATHS['carinthia'], image_size=32, force=False
)

PosixPath('/root/xxx/autodl/data_cache/carinthia_32.npz')

## 2. 固定论文配置

该配置对四种模型完全一致。数据集适配器只会调整输入通道数和输出类别数。正式比较前不要根据测试结果修改配置。

In [35]:
BASE_CONFIG = replace(
    ExperimentConfig.publication(),
    quantum_projection_mode='five',
    epochs=60,
    patience=10,
    batch_size=64,
    sampler_power=0.5,
    quantum_init_scale=0.1,
    train_cap_per_class=2000,
    eval_cap_per_class=None,
    num_workers=0,
)
SEEDS = (42, 52, 62, 72, 82)
SPLIT_SEED = 2026

# 每次只运行一个数据集：wm811k / mixedwm38 / carinthia
DATASET_TO_RUN = 'carinthia'
CACHE_PATH = CACHE_PATHS[DATASET_TO_RUN]
print(DATASET_TO_RUN, CACHE_PATH)

carinthia /root/xxx/autodl/data_cache/carinthia_32.npz


## 3. 数据、划分、参数量与梯度审计

In [ ]:
bundle = load_dataset_cache(CACHE_PATH)
split = make_dataset_split(bundle, seed=SPLIT_SEED)
print(bundle.describe())
display(class_distribution(bundle).round(3))
display(split_distribution(bundle, split).pivot(index='label', columns='split', values='count'))

audit = dataset_audit(
    CACHE_PATH, BASE_CONFIG, split_seed=SPLIT_SEED, run_gradient_test=True
)
display(audit['parameter_audit'].round(3))
display(audit['gradient_test'].round(3))

{'dataset': 'mixedwm38', 'samples': 38015, 'image_shape': (32, 32), 'image_dtype': 'uint8', 'classes': 38, 'groups': 38015, 'input_kind': 'wafer_map', 'split_strategy': 'stratified', 'cache_path': '/root/xxx/autodl/data_cache/mixedwm38_32.npz'}


,class_id,label,count,fraction
0,0,none,1000,0.026
1,1,random,866,0.023
2,2,scratch,1000,0.026
3,3,near-full,149,0.004
4,4,loc,1000,0.026
5,5,edge-ring,1000,0.026
6,6,edge-loc,1000,0.026
7,7,donut,1000,0.026
8,8,center,1000,0.026
9,9,loc+scratch,1000,0.026


split,test,train,val
label,,,
center,150,700,150
center+edge-loc,150,700,150
center+edge-loc+loc,150,700,150
center+edge-loc+loc+scratch,150,700,150
center+edge-loc+scratch,300,1400,300
center+edge-ring,150,700,150
center+edge-ring+loc,150,700,150
center+edge-ring+loc+scratch,150,700,150
center+edge-ring+scratch,150,700,150


,model,parameters,relative_to_quantum
0,quantum_transformer,2446,0.000
1,tiny_transformer,2502,0.023
2,mlp_mixer,2504,0.024
3,cnn_token_mixer,2462,0.007


,model,logit_shape,loss,missing_gradients,all_gradients_finite
0,quantum_transformer,"(2, 38)",4.320,0,True
1,tiny_transformer,"(2, 38)",3.806,0,True
2,mlp_mixer,"(2, 38)",3.918,0,True
3,cnn_token_mixer,"(2, 38)",3.758,0,True


## 4. 可选：单种子短试跑

第一次接入某个数据集时先运行本单元。结果仅用于排错，不进入论文。

In [ ]:
SMOKE_CONFIG = replace(
    BASE_CONFIG, epochs=2, patience=2, train_cap_per_class=16, eval_cap_per_class=16
)
smoke_results, _, _ = run_comparison_suite(
    cache_path=CACHE_PATH,
    base_config=SMOKE_CONFIG,
    seeds=(42,),
    split_seed=SPLIT_SEED,
    artifact_dir=PROJECT_DIR / 'artifacts' / 'smoke_three_datasets',
    resume=True,
)
display(smoke_results.round(3))

,dataset,model,seed,parameters,train_seconds,best_epoch,best_val_macro_f1,train_samples,val_samples,test_samples,...,recall_donut+edge-loc+loc,recall_center+loc+scratch,recall_center+edge-ring+scratch,recall_center+edge-ring+loc,recall_center+edge-loc+scratch,recall_center+edge-loc+loc,recall_donut+edge-ring+loc+scratch,recall_donut+edge-loc+loc+scratch,recall_center+edge-ring+loc+scratch,recall_center+edge-loc+loc+scratch
0,mixedwm38,cnn_token_mixer,42,2462,0.274,2,0.002,608,608,608,...,0.0,0.062,0.0,0.0,0.000,0.0,0.000,0.875,0.000,0.0
1,mixedwm38,mlp_mixer,42,2504,0.307,1,0.010,608,608,608,...,0.0,1.000,0.0,0.0,0.000,0.0,0.000,0.000,0.000,0.0
2,mixedwm38,quantum_transformer,42,2446,2.393,2,0.010,608,608,608,...,0.0,0.000,0.0,0.0,0.562,0.0,0.438,0.000,0.688,0.0
3,mixedwm38,tiny_transformer,42,2502,0.312,2,0.005,608,608,608,...,0.0,0.000,0.0,0.0,0.000,0.0,0.000,0.000,0.000,0.0


In [ ]:
from pathlib import Path

CACHE_PATH = CACHE_PATHS["mixedwm38"]

SEEDS = (42, 52, 62, 72, 82)
SPLIT_SEED = 2026

print("CACHE_PATH:", CACHE_PATH)
print("ARTIFACT_ROOT:", ARTIFACT_ROOT)

CACHE_PATH: /root/xxx/autodl/data_cache/mixedwm38_32.npz
ARTIFACT_ROOT: /root/xxx/autodl/artifacts/three_datasets_five_projection


In [ ]:
from pathlib import Path
import pandas as pd

MODEL_NAMES = (
    "quantum_transformer",
    "tiny_transformer",
    "mlp_mixer",
    "cnn_token_mixer",
)

# run_comparison_suite 会自动在 ARTIFACT_ROOT 后添加数据集名称
RUN_DIR = Path(ARTIFACT_ROOT) / "mixedwm38"

required_files = (
    "signature.json",
    "result.json",
    "history.json",
    "confusion.npy",
    "best.pt",
)

status_rows = []

for seed in SEEDS:
    for model in MODEL_NAMES:
        job_dir = RUN_DIR / model / f"seed_{seed}"
        missing = [
            name for name in required_files
            if not (job_dir / name).exists()
        ]

        status_rows.append({
            "seed": seed,
            "model": model,
            "status": "已完成" if not missing else "需要续跑",
            "missing": ", ".join(missing),
            "directory": str(job_dir),
        })

status_df = pd.DataFrame(status_rows)
display(status_df[["seed", "model", "status", "missing"]])

print("\n完成数量：", (status_df["status"] == "已完成").sum(), "/ 20")
print("待运行数量：", (status_df["status"] != "已完成").sum(), "/ 20")

,seed,model,status,missing
0,42,quantum_transformer,已完成,
1,42,tiny_transformer,已完成,
2,42,mlp_mixer,已完成,
3,42,cnn_token_mixer,已完成,
4,52,quantum_transformer,已完成,
5,52,tiny_transformer,已完成,
6,52,mlp_mixer,已完成,
7,52,cnn_token_mixer,已完成,
8,62,quantum_transformer,已完成,
9,62,tiny_transformer,已完成,



完成数量： 20 / 20
待运行数量： 0 / 20


## 5. 正式五种子比较

该单元可断点续跑。建议依次完成 `mixedwm38`、`carinthia`，最后在需要时用相同新管线复核 `wm811k`。

In [ ]:
results, histories, confusion_matrices = run_comparison_suite(
    cache_path=CACHE_PATH,
    base_config=BASE_CONFIG,
    seeds=SEEDS,
    split_seed=SPLIT_SEED,
    artifact_dir=ARTIFACT_ROOT,
    resume=True,
)
display(results.round(3))
display(summarize_results(results).round(3))

,dataset,model,seed,parameters,train_seconds,best_epoch,best_val_macro_f1,train_samples,val_samples,test_samples,...,recall_donut+edge-loc+loc,recall_center+loc+scratch,recall_center+edge-ring+scratch,recall_center+edge-ring+loc,recall_center+edge-loc+scratch,recall_center+edge-loc+loc,recall_donut+edge-ring+loc+scratch,recall_donut+edge-loc+loc+scratch,recall_center+edge-ring+loc+scratch,recall_center+edge-loc+loc+scratch
0,mixedwm38,cnn_token_mixer,42,2462,217.652,47,0.424,26610,5702,5703,...,0.447,0.493,0.253,0.213,0.820,0.047,0.747,0.347,0.553,0.307
1,mixedwm38,mlp_mixer,42,2504,233.918,55,0.428,26610,5702,5703,...,0.233,0.493,0.467,0.107,0.300,0.153,0.653,0.420,0.620,0.280
2,mixedwm38,quantum_transformer,42,2446,1846.881,37,0.427,26610,5702,5703,...,0.287,0.580,0.520,0.440,0.620,0.207,0.387,0.373,0.193,0.267
3,mixedwm38,tiny_transformer,42,2502,219.950,41,0.419,26610,5702,5703,...,0.367,0.500,0.227,0.300,0.663,0.173,0.333,0.220,0.527,0.193
4,mixedwm38,cnn_token_mixer,52,2462,349.292,57,0.432,26610,5702,5703,...,0.387,0.460,0.253,0.407,0.680,0.400,0.740,0.233,0.540,0.200
5,mixedwm38,mlp_mixer,52,2504,382.081,59,0.459,26610,5702,5703,...,0.227,0.140,0.753,0.247,0.693,0.387,0.600,0.447,0.547,0.187
6,mixedwm38,quantum_transformer,52,2446,2420.346,56,0.452,26610,5702,5703,...,0.267,0.327,0.493,0.327,0.813,0.187,0.700,0.407,0.447,0.113
7,mixedwm38,tiny_transformer,52,2502,433.803,44,0.427,26610,5702,5703,...,0.627,0.667,0.500,0.373,0.570,0.113,0.413,0.293,0.353,0.527
8,mixedwm38,cnn_token_mixer,62,2462,413.507,52,0.452,26610,5702,5703,...,0.573,0.513,0.273,0.420,0.633,0.240,0.767,0.407,0.573,0.200
9,mixedwm38,mlp_mixer,62,2504,447.264,53,0.413,26610,5702,5703,...,0.213,0.380,0.620,0.273,0.793,0.000,0.800,0.360,0.533,0.140


,macro_f1_mean,macro_f1_std,balanced_acc_mean,balanced_acc_std,parameters,train_seconds_mean
model,,,,,,
cnn_token_mixer,0.438,0.010,0.458,0.012,2462,348.957
mlp_mixer,0.435,0.009,0.454,0.006,2504,344.582
tiny_transformer,0.423,0.007,0.442,0.010,2502,434.036
quantum_transformer,0.395,0.052,0.414,0.046,2446,2818.683


In [ ]:
from pathlib import Path
import pandas as pd

SEEDS = (42, 52, 62, 72, 82)

from pathlib import Path

# run_comparison_suite 会自动在 ARTIFACT_ROOT 下添加数据集名称
RUN_DIR = Path(ARTIFACT_ROOT) / "mixedwm38"

print("正式实验目录：", RUN_DIR)
print("目录是否存在：", RUN_DIR.exists())

completed_seeds = []
pending_seeds = []

for seed in SEEDS:
    seed_dir = RUN_DIR / f"seed_{seed}"
    result_file = seed_dir / "comparison_results.csv"

    if result_file.exists():
        try:
            result = pd.read_csv(result_file)
            finished_models = set(result["model"].astype(str))
            required_models = {
                "quantum_transformer",
                "tiny_transformer",
                "mlp_mixer",
                "cnn_token_mixer",
            }

            if required_models.issubset(finished_models):
                completed_seeds.append(seed)
            else:
                pending_seeds.append(seed)
        except Exception:
            pending_seeds.append(seed)
    else:
        pending_seeds.append(seed)

print("已经完成：", completed_seeds)
print("需要续跑：", pending_seeds)

正式实验目录： /root/xxx/autodl/artifacts/three_datasets_five_projection/mixedwm38
目录是否存在： True
已经完成： []
需要续跑： [42, 52, 62, 72, 82]


## 6. 三数据集完成后汇总

若某个数据集尚未完成，本单元会显示缺失项，不会伪造或填补结果。

In [ ]:
frames = []
missing = []
for dataset_name in ('wm811k', 'mixedwm38', 'carinthia'):
    result_file = ARTIFACT_ROOT / dataset_name / 'comparison_results.csv'
    if result_file.exists():
        frame = pd.read_csv(result_file)
        expected = len(SEEDS) * 4
        if len(frame) == expected:
            frames.append(frame)
        else:
            missing.append(f'{dataset_name}: {len(frame)}/{expected} jobs')
    else:
        missing.append(f'{dataset_name}: no result file')

print('Missing:', missing if missing else 'none')
if not missing:
    all_results = pd.concat(frames, ignore_index=True)
    all_results.to_csv(ARTIFACT_ROOT / 'three_dataset_results.csv', index=False)
    display(all_results.groupby(['dataset', 'model'])[['macro_f1', 'balanced_accuracy']].agg(['mean', 'std']).round(3))

Missing: ['wm811k: no result file', 'carinthia: no result file']


In [ ]:
if not missing:
    paired = paired_model_comparisons(all_results, metric='macro_f1')
    paired.to_csv(ARTIFACT_ROOT / 'paired_macro_f1.csv', index=False)
    display(paired.round(3))

In [ ]:
if not missing:
    figure = plot_cross_dataset_results(all_results, metric='macro_f1')
    figure_dir = PROJECT_DIR / 'paper' / 'figures'
    figure_dir.mkdir(parents=True, exist_ok=True)
    figure.savefig(figure_dir / 'three_dataset_macro_f1.png', bbox_inches='tight', dpi=300)
    figure.savefig(figure_dir / 'three_dataset_macro_f1.pdf', bbox_inches='tight')
    plt.show()

## 7. 可选：已保存检查点的噪声鲁棒性

晶圆图噪声表示缺陷位翻转概率；Carinthia 噪声表示归一化灰度图上的高斯噪声标准差。两种噪声不能直接解释为同一种物理扰动。

In [ ]:
noise_results = evaluate_noise_suite(
    cache_path=CACHE_PATH,
    artifact_dir=ARTIFACT_ROOT,
    seeds=SEEDS,
    noise_levels=(0.00, 0.01, 0.03, 0.05, 0.10),
    split_seed=SPLIT_SEED,
)
display(noise_results.round(3))

,dataset,model,seed,noise,noise_kind,macro_f1,balanced_accuracy,clean_macro_f1,macro_f1_retention
0,mixedwm38,quantum_transformer,42,0.00,defect_bit_flip,0.424,0.438,0.424,1.000
1,mixedwm38,quantum_transformer,42,0.01,defect_bit_flip,0.404,0.417,0.424,0.953
2,mixedwm38,quantum_transformer,42,0.03,defect_bit_flip,0.375,0.384,0.424,0.883
3,mixedwm38,quantum_transformer,42,0.05,defect_bit_flip,0.331,0.342,0.424,0.781
4,mixedwm38,quantum_transformer,42,0.10,defect_bit_flip,0.209,0.236,0.424,0.492
...,...,...,...,...,...,...,...,...,...
95,mixedwm38,cnn_token_mixer,82,0.00,defect_bit_flip,0.438,0.461,0.438,1.000
96,mixedwm38,cnn_token_mixer,82,0.01,defect_bit_flip,0.432,0.454,0.438,0.986
97,mixedwm38,cnn_token_mixer,82,0.03,defect_bit_flip,0.403,0.426,0.438,0.919
98,mixedwm38,cnn_token_mixer,82,0.05,defect_bit_flip,0.360,0.382,0.438,0.821


In [ ]:
key_columns = [
    "model",
    "seed",
    "parameters",
    "best_epoch",
    "best_val_macro_f1",
    "accuracy",
    "balanced_accuracy",
    "macro_precision",
    "macro_recall",
    "macro_f1",
    "train_seconds",
]

display(
    results[key_columns]
    .sort_values(["model", "seed"])
    .round(3)
)

summary = (
    results.groupby("model")
    .agg(
        val_macro_f1_mean=("best_val_macro_f1", "mean"),
        val_macro_f1_std=("best_val_macro_f1", "std"),
        test_macro_f1_mean=("macro_f1", "mean"),
        test_macro_f1_std=("macro_f1", "std"),
        balanced_acc_mean=("balanced_accuracy", "mean"),
        balanced_acc_std=("balanced_accuracy", "std"),
        train_seconds_mean=("train_seconds", "mean"),
        parameters=("parameters", "first"),
    )
    .sort_values("test_macro_f1_mean", ascending=False)
)

display(summary.round(3))

,model,seed,parameters,best_epoch,best_val_macro_f1,accuracy,balanced_accuracy,macro_precision,macro_recall,macro_f1,train_seconds
0,cnn_token_mixer,42,2462,47,0.424,0.452,0.439,0.446,0.439,0.422,217.652
4,cnn_token_mixer,52,2462,57,0.432,0.447,0.454,0.438,0.454,0.438,349.292
8,cnn_token_mixer,62,2462,52,0.452,0.462,0.471,0.448,0.471,0.449,413.507
12,cnn_token_mixer,72,2462,51,0.444,0.461,0.462,0.447,0.462,0.444,426.168
16,cnn_token_mixer,82,2462,52,0.441,0.446,0.461,0.432,0.461,0.438,338.168
1,mlp_mixer,42,2504,55,0.428,0.447,0.452,0.436,0.452,0.428,233.918
5,mlp_mixer,52,2504,59,0.459,0.467,0.463,0.460,0.463,0.448,382.081
9,mlp_mixer,62,2504,53,0.413,0.451,0.453,0.422,0.453,0.424,447.264
13,mlp_mixer,72,2504,48,0.443,0.440,0.453,0.442,0.453,0.436,443.567
17,mlp_mixer,82,2504,32,0.435,0.451,0.448,0.456,0.448,0.438,216.082


,val_macro_f1_mean,val_macro_f1_std,test_macro_f1_mean,test_macro_f1_std,balanced_acc_mean,balanced_acc_std,train_seconds_mean,parameters
model,,,,,,,,
cnn_token_mixer,0.438,0.011,0.438,0.010,0.458,0.012,348.957,2462
mlp_mixer,0.436,0.017,0.435,0.009,0.454,0.006,344.582,2504
tiny_transformer,0.422,0.011,0.423,0.007,0.442,0.010,434.036,2502
quantum_transformer,0.397,0.050,0.395,0.052,0.414,0.046,2818.683,2446
